# Predicting the Energy Performance of French Houses

## 1. Project & Dataset Description

**Aim.** Predict the energy performance of a French house from its physical characteristics only
(surface, construction period, insulation, heating system, climate zone...), as two tasks:

- **Regression**: annual conventional primary energy consumption `conso_5_usages_par_m2_ep` (kWh/m²/year)
- **Classification**: energy label `etiquette_dpe` (7 classes, A to G)

**Dataset.** [DPE Logements existants (depuis juillet 2021)](https://data.ademe.fr/datasets/dpe03existant), ADEME,
Licence Ouverte 2.0, also on [data.gouv.fr](https://www.data.gouv.fr/datasets/dpe-logements-existants-depuis-juillet-2021).
Our fixed sample contains 80,000 house diagnostics from region 84, extracted on 2026-11-17.
It contains 42 stored fields, of which 17 are selected as model inputs. Duplicate diagnosis IDs
and known replaced diagnoses were removed during extraction; several diagnoses may still concern
the same building. The common preparation and grouped split are documented in section 5.1.

**Existing solutions.** TODO: summarise the related work listed in the team form and how we differ.

## 2. Imports

Colab already provides these libraries. Versions used are printed below for reproducibility.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

SEED = 42
np.random.seed(SEED)

print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

pandas 2.2.2 | scikit-learn 1.4.2


## 3. Data Access

L'échantillon de 80 000 diagnostics et son manifeste sont déjà disponibles sur GitHub.
La cellule suivante lit une version fixe : il n'est pas nécessaire de relancer l'extraction ADEME.
Le bloc d'extraction commenté est conservé uniquement pour documenter la provenance.

In [2]:
# !git clone https://github.com/Yannisloum63/dpe-house-energy-prediction.git
# %cd dpe-house-energy-prediction
# !python scripts/extract_dpe.py --region 84 --n 80000

In [3]:
import requests

# Version fixe du jeu de données
VERSION = "f47d6754403fc71d7a5313292785ba472571ba21"
BASE_URL = (
    "https://raw.githubusercontent.com/"
    f"Yannisloum63/dpe-house-energy-prediction/{VERSION}/data/"
)

df = pd.read_csv(
    BASE_URL + "dpe_houses.csv.gz",
    compression="gzip",
    dtype={
        "numero_dpe": "string",
        "numero_dpe_remplace": "string",
        "id_rnb": "string",
        "code_insee_ban": "string",
        "code_departement_ban": "string",
        "code_region_ban": "string",
    },
)

response = requests.get(
    BASE_URL + "dpe_houses.csv.gz.manifest.json", timeout=30
)
response.raise_for_status()
manifest = response.json()

assert len(df) == manifest["saved_rows"] == 80000
print("Dimensions :", df.shape)
df.head()

Dimensions : (80000, 42)


,numero_dpe,numero_dpe_remplace,date_etablissement_dpe,code_insee_ban,code_departement_ban,code_region_ban,coordonnee_cartographique_x_ban,coordonnee_cartographique_y_ban,surface_habitable_logement,annee_construction,...,conso_5_usages_par_m2_ep,etiquette_dpe,emission_ges_5_usages_par_m2,etiquette_ges,type_installation_chauffage_n1,type_batiment,id_rnb,version_dpe,type_installation_chauffage_n2,ventilation_posterieure_2012
0,2238E0355918R,2138E0546906I,2022-02-22,38087,38,84,839981.79,6499791.42,79.0,1948.0,...,251.6,E,10.0,B,installation individuelle,maison,83HHVD177CD4,2.0,NaN,0
1,2269E1277295L,<NA>,2022-06-10,69094,69,84,831211.13,6517579.14,138.2,1988.0,...,186.7,D,43.0,D,installation individuelle,maison,ZTAPMG7FVH7Z,2.1,NaN,0
2,2574E3225392O,<NA>,2025-10-12,74062,74,84,944421.21,6547447.15,175.0,1970.0,...,508.7,G,110.0,G,installation individuelle,maison,<NA>,2.5,NaN,0
3,2573E2135618D,<NA>,2025-06-28,73235,73,84,962533.62,6485003.01,49.6,NaN,...,398.6,F,15.0,C,installation individuelle,maison,S4Y21TCKBW1S,2.4,NaN,0
4,2426E0103928J,2426E0097882V,2024-01-11,26054,26,84,849694.89,6357000.43,125.0,1990.0,...,211.7,D,8.0,B,installation individuelle,maison,<NA>,2.3,NaN,0


### Columns that must never be model inputs

The label is the worse of the consumption class and the GHG emission class, so the computed outputs leak the targets.

In [4]:
TARGET_REG = "conso_5_usages_par_m2_ep"
TARGET_CLF = "etiquette_dpe"

# Les 17 variables retenues ; les ensembles seront chargés en section 5.1.
FEATURES = manifest["feature_columns"]
assert len(FEATURES) == 17
assert all(column in df.columns for column in FEATURES)
print("Variables d'entrée :", df[FEATURES].shape)

Variables d'entrée : (80000, 17)


## 4. Exploratory Data Analysis

Ces premiers contrôles descriptifs ont été réalisés sur les 80 000 lignes avant la séparation.
Ils sont conservés comme état initial. Pour les nouvelles analyses qui orientent les modèles,
exécuter la section 5.1 puis utiliser `df_train` ; ne pas explorer le test pour choisir un modèle.

### 4.1 Metadata: number of instances, features, types

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 42 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   numero_dpe                                      80000 non-null  string 
 1   numero_dpe_remplace                             7299 non-null   string 
 2   date_etablissement_dpe                          80000 non-null  object 
 3   code_insee_ban                                  80000 non-null  string 
 4   code_departement_ban                            80000 non-null  string 
 5   code_region_ban                                 80000 non-null  string 
 6   coordonnee_cartographique_x_ban                 80000 non-null  float64
 7   coordonnee_cartographique_y_ban                 80000 non-null  float64
 8   surface_habitable_logement                      80000 non-null  float64
 9   annee_construction                     

### 4.2 Missing values: identification, interpretation, chosen strategy

In [6]:
df[FEATURES].isna().mean().sort_values(
    ascending=False
).to_frame("missing_rate")

,missing_rate
isolation_toiture,0.075237
type_ventilation,0.062100
type_installation_chauffage_n1,0.036375
type_emetteur_installation_chauffage_n1,0.036375
classe_inertie_batiment,0.010212
qualite_isolation_plancher_bas,0.009000
zone_climatique,0.002350
classe_altitude,0.002350
qualite_isolation_murs,0.000013
periode_construction,0.000000


### 4.3 Feature distributions, scaling and outliers

In [7]:
df.describe()

,coordonnee_cartographique_x_ban,coordonnee_cartographique_y_ban,surface_habitable_logement,annee_construction,hauteur_sous_plafond,nombre_niveau_logement,isolation_toiture,type_installation_chauffage,presence_production_pv,surface_totale_capteurs_pv,conso_5_usages_par_m2_ep,emission_ges_5_usages_par_m2,version_dpe,ventilation_posterieure_2012
count,8.000000e+04,8.000000e+04,80000.000000,40996.000000,80000.000000,80000.000000,73981.000000,0.0,5738.000000,2855.000000,80000.000000,80000.000000,80000.000000,80000.000000
mean,8.309366e+05,6.496244e+06,117.192889,1974.216753,2.555823,1.793538,0.871629,NaN,0.718892,16.707741,241.629307,30.125663,2.262723,0.219138
std,8.620845e+04,2.180599e+05,60.327943,35.963063,2.354796,1.121049,0.334504,NaN,0.449579,55.443813,130.386524,33.247984,0.342281,0.413665
min,0.000000e+00,0.000000e+00,7.000000,1500.000000,1.000000,1.000000,0.000000,NaN,0.000000,0.500000,22.100000,0.000000,1.000000,0.000000
25%,7.818210e+05,6.467819e+06,83.600000,1949.000000,2.500000,1.000000,1.000000,NaN,0.000000,8.400000,153.000000,7.000000,2.200000,0.000000
50%,8.408752e+05,6.511134e+06,104.100000,1979.000000,2.500000,2.000000,1.000000,NaN,1.000000,14.400000,215.500000,15.000000,2.300000,0.000000
75%,8.863233e+05,6.543315e+06,137.200000,2001.000000,2.500000,2.000000,1.000000,NaN,1.000000,19.200000,301.600000,44.000000,2.400000,0.000000
max,1.016806e+06,6.631829e+06,3771.000000,2026.000000,282.000000,184.000000,1.000000,NaN,1.000000,2880.000000,2128.200000,652.700000,2.600000,1.000000


In [8]:
# Examiner les valeurs extrêmes avant toute correction
fields = [
    "surface_habitable_logement",
    "hauteur_sous_plafond",
    "nombre_niveau_logement",
    TARGET_REG,
]

columns = ["numero_dpe", "periode_construction"] + fields + [
    TARGET_CLF, "version_dpe"
]

for field in fields:
    print(f"\nLes 5 valeurs les plus élevées : {field}")
    display(df.nlargest(5, field)[columns])


Les 5 valeurs les plus élevées : surface_habitable_logement


,numero_dpe,periode_construction,surface_habitable_logement,hauteur_sous_plafond,nombre_niveau_logement,conso_5_usages_par_m2_ep,etiquette_dpe,version_dpe
4914,2474E1371781U,2013-2021,3771.0,2.5,3.0,46.9,A,2.3
71382,2373E1590359O,1989-2000,3000.0,2.5,5.0,68.3,C,2.2
30977,2174E0992873H,1948-1974,2272.0,2.5,1.0,168.1,C,2.0
36470,2201E1436878Q,avant 1948,1450.0,3.0,1.0,226.7,D,2.1
47809,2469E2086416O,avant 1948,1316.8,3.7,7.0,255.3,E,2.3



Les 5 valeurs les plus élevées : hauteur_sous_plafond


,numero_dpe,periode_construction,surface_habitable_logement,hauteur_sous_plafond,nombre_niveau_logement,conso_5_usages_par_m2_ep,etiquette_dpe,version_dpe
40381,2501E1674693M,avant 1948,174.4,282.0,2.0,358.5,F,2.4
39258,2638E0640397I,avant 1948,214.2,272.0,3.0,255.7,E,2.6
45170,2171E0805451Q,avant 1948,185.9,256.0,2.0,255.7,E,1.1
5612,2138E0566399B,2006-2012,102.9,246.0,1.0,155.0,C,1.0
78938,2503E2985294H,avant 1948,42.5,242.0,1.0,628.0,G,2.5



Les 5 valeurs les plus élevées : nombre_niveau_logement


,numero_dpe,periode_construction,surface_habitable_logement,hauteur_sous_plafond,nombre_niveau_logement,conso_5_usages_par_m2_ep,etiquette_dpe,version_dpe
70021,2242E0858240X,avant 1948,184.0,2.6,184.0,244.5,F,2.0
9390,2601E2721538U,avant 1948,210.0,4.0,115.0,73.0,B,2.6
16365,2103E0606150P,1948-1974,167.7,2.5,80.0,388.9,F,1.1
35977,2626E0827918F,avant 1948,119.0,2.5,66.0,232.0,D,2.6
44325,2674E0367140V,2006-2012,134.1,2.5,46.0,178.0,C,2.6



Les 5 valeurs les plus élevées : conso_5_usages_par_m2_ep


,numero_dpe,periode_construction,surface_habitable_logement,hauteur_sous_plafond,nombre_niveau_logement,conso_5_usages_par_m2_ep,etiquette_dpe,version_dpe
14424,2338E0125716U,avant 1948,310.0,3.4,4.0,2128.2,G,2.2
62803,2574E1892226Y,1948-1974,150.8,2.6,3.0,2064.8,G,2.4
3390,2273E1880525T,avant 1948,70.0,2.5,1.0,1693.9,G,2.1
55955,2569E1741556T,avant 1948,7.0,2.5,1.0,1556.0,G,2.4
75701,2174E0316175L,1948-1974,152.0,2.7,3.0,1538.6,G,1.0


### 4.4 Target study: distribution and class balance

In [9]:
df[TARGET_CLF].value_counts(normalize=True).sort_index()

etiquette_dpe
A    0.032175
B    0.050888
C    0.217950
D    0.286050
E    0.195025
F    0.112025
G    0.105887
Name: proportion, dtype: float64

### 4.5 Correlations, feature selection, dimension reduction, anomaly detection

### 4.6 Unsupervised clustering and correlation with the targets

### 4.7 Data enrichment from other sources (INSEE, Météo-France...)

### 4.8 Interpretations, hypotheses and conclusions

## 5. Baseline & Ensemble Models

### 5.1 Préparation commune et séparation fixe

La source `df` reste intacte. Dans une copie, les hauteurs > 10 m et les nombres de niveaux > 10
sont mis à NaN, sans deviner une conversion d'unité et sans supprimer de ligne. Les valeurs non
positives/non finies, les niveaux non entiers et les indicateurs de toit hors {0,1} sont aussi
contrôlés. Les seuils sont une convention prudente du projet, pas une norme officielle.
Cette version masque 24 cellules sur 24 diagnostics (13 hauteurs et 11 nombres de niveaux).
Les grandes surfaces et les consommations élevées sont conservées.

Les diagnostics liés par un RNB connu ou une référence de remplacement restent ensemble.
Une séparation stratifiée par groupes a été générée une seule fois avec la graine 42, puis
publiée : 47 974 train, 16 007 validation et 16 019 test. Les cellules ci-dessous réutilisent
ces affectations ; elles ne font pas de nouveau tirage.

**Limites :** 40 698 RNB manquants, plusieurs versions DPE non harmonisées, statistiques
descriptives déjà consultées avant la séparation. L'absence de chevauchement des groupes connus
ne garantit donc pas une indépendance parfaite entre bâtiments.

L'imputation, l'encodage et la standardisation seront ajustés dans les plis d'entraînement du
Pipeline. Réutiliser `groups_train` lors de la validation croisée. Le test reste réservé à la fin.
[Documentation de la préparation](https://github.com/Yannisloum63/dpe-house-energy-prediction/blob/fa295ee53d4f8845a0076979dfed0d32da712122/docs/data_preparation.md).

In [10]:
import hashlib
import importlib.util
from pathlib import Path

# Code et séparation figés pour toute l’équipe.
PREPARATION_VERSION = "fa295ee53d4f8845a0076979dfed0d32da712122"
PREPARATION_URL = (
    "https://raw.githubusercontent.com/"
    f"Yannisloum63/dpe-house-energy-prediction/{PREPARATION_VERSION}"
)
response = requests.get(PREPARATION_URL + "/scripts/prepare_data.py", timeout=60)
response.raise_for_status()
assert hashlib.sha256(response.content).hexdigest() == "fc5da6179798ab70688a40b1f38cdabea2509df017756916ed8a4ea198cc9628"
module_path = Path("prepare_data_shared.py")
module_path.write_bytes(response.content)
spec = importlib.util.spec_from_file_location("prepare_data_shared", module_path)
preparation = importlib.util.module_from_spec(spec)
spec.loader.exec_module(preparation)

In [11]:
df_raw, datasets, FEATURES, masked_values, preparation_report = preparation.load_shared_data(PREPARATION_URL)
df_train = datasets["train"]
df_val = datasets["validation"]
df_test = datasets["test"]

# Diagnostic de préparation, sans entraînement de modèle.
display(pd.DataFrame({
    "ensemble": ["train", "validation", "test"],
    "lignes": [len(df_train), len(df_val), len(df_test)],
    "variables": [len(FEATURES)] * 3,
}))
display(masked_values.groupby("field").size().to_frame("valeurs_masquees"))
print("Groupes connus partagés :", preparation_report["cross_split_groups"])
print("RNB manquants :", preparation_report["rnb_missing"])

,ensemble,lignes,variables
0,train,47974,17
1,validation,16007,17
2,test,16019,17


,valeurs_masquees
field,
hauteur_sous_plafond,13
nombre_niveau_logement,11


Groupes connus partagés : 0
RNB manquants : 40698


In [12]:
X_train = df_train[FEATURES].copy()
X_val = df_val[FEATURES].copy()
X_test = df_test[FEATURES].copy()

y_train_reg, y_val_reg, y_test_reg = (
    frame[TARGET_REG].copy() for frame in [df_train, df_val, df_test]
)
y_train_clf, y_val_clf, y_test_clf = (
    frame[TARGET_CLF].copy() for frame in [df_train, df_val, df_test]
)
groups_train = df_train["group_id"].copy()
NUMERIC_FEATURES = preparation_report["numeric_features"]
CATEGORICAL_FEATURES = preparation_report["categorical_features"]

print("Prêt pour les graphiques sur df_train et les modèles sur X_train / X_val.")

Prêt pour les graphiques sur df_train et les modèles sur X_train / X_val.


**Pour l'équipe :** utiliser `df_train` pour les nouvelles analyses ; entraîner les modèles avec
`X_train` et les cibles correspondantes, puis évaluer sur `X_val`. Les entrées contiennent encore
des valeurs manquantes : les traiter dans le Pipeline, sans ajustement sur validation/test.
`group_id`, `split` et les autres identifiants ne font pas partie des 17 entrées.
Aucun modèle n'a encore été entraîné dans cette étape.

### 5.2 Preprocessing pipelines and models

### 5.3 Training and validation (nested cross-validation, learning curves)

### 5.4 Test and interpretation of results

## 6. Enhanced Models & Hyperparameter Tuning

### 6.1 Justification of choices

### 6.2 Tuning (parallel coordinates plot of the search)

### 6.3 Interpretation and analysis of the results

## 7. Conclusion and outlook